## **O problema do McBurger**
O restaurante de fast-food McBurger vende quarteirões e cheeseburgers. Um quarteirão custa 1/4 (0,25) de libra de carne e um chesseburger usa apenas 0,2 libra. O restaurante começa o dia com 250 libras de carne, mas pode pedir mais carne a um custo adicional de 28 centavos por libra, para cobrir o custo de entrega. Qualquer excedente de carne ao final do dia é doado para a caridade.

O lucro do McBurger é de 22 centavos por quarteirão e 18 centavos por cheeseburger. O McBurger não espera vender mais de 950 sanduíches por dia. 

Quantos sanduíches de cada tipo o McBurger deve planejar produzir por dia? Resolva o problema manualmente e usando AMPL.

## Vamos resolver o problema manualmente e depois via AMPL:

### **1. Variáveis de decisão:**
q -> quantidade de quarteirões
c -> quantidade de cheeseburgers
x -> quantidade de libras de carne adicionais

**1.1. Receita:**
R(x) = 0,22q + 0,18c

**1.2. Custo:**
C(x) = 0,28x

**1.3. Vendas:**
q + c <= 950

**2. Função objetivo: (Lucro = Receita - Custo)** <br>
max Z = 0,22q + 0,18c - 0,28x

**3. Restrições:**
1. Disponibilidade de carne: 0,25q + 0,2c <= 250 + x
2. Limite de produção: q + c <= 950
3. Não negatividade: q >= 0, c >= 0, x >= 0


In [6]:
# 1. Instalação
%pip install -U -q amplpy
import sys, subprocess

subprocess.check_call([sys.executable, "-m", "amplpy.modules", "install", "--upgrade", "open"])

Note: you may need to restart the kernel to use updated packages.
$ /home/m9t/Documents/ufg/o-research/.venv/bin/python -m pip install -i https://pypi.ampl.com ampl_module_base ampl_module_open --upgrade
Looking in indexes: https://pypi.ampl.com
Imported ampl_module_base.
Imported ampl_module_base.
Imported ampl_module_open.


0

In [7]:
# 2. PATH do AMPL e importação
import os, sys, subprocess

caminho_mod = subprocess.check_output([sys.executable, "-m", "amplpy.modules", "path"]).decode().strip()
os.environ["PATH"] += os.pathsep + caminho_mod

from amplpy import AMPL

print("AMPL pronto (solver padrão configurado por modelo abaixo).")

AMPL pronto (solver padrão configurado por modelo abaixo).


In [11]:
def modelo_mcburger():
    """Modelo AMPL do problema do McBurger."""
    ampl = AMPL()
    ampl.eval("option solver cbc;")
    ampl.eval(
        """

        var q >= 0;  # quantidade de quarteiroes
        var c >= 0;  # quantidade de cheeseburgers
        var x >= 0;  # carne adicional comprada (libras)

        maximize z: 0.22 * q + 0.18 * c - 0.28 * x;

        subject to limite_producao:
            q + c <= 950;

        subject to disponibilidade_carne:
            0.25 * q + 0.2 * c <= 250 + x;
        """
    )
    ampl.solve()
    ampl.eval("display z, q, c, x;")
    ampl.eval("display (250 + x - 0.25*q - 0.2*c);")

    quantidade_de_quarteiroes = ampl.get_value("q")
    quantidade_de_cheeseburgers = ampl.get_value("c")
    quantidade_de_libras_de_carne_adicionais = ampl.get_value("x")
    lucro = ampl.get_value("z")

    carne_inicial = 250
    carne_usada = 0.25 * quantidade_de_quarteiroes + 0.2 * quantidade_de_cheeseburgers
    carne_disponivel = carne_inicial + quantidade_de_libras_de_carne_adicionais
    excedente_carne = carne_disponivel - carne_usada

    print(f"Quantidade de quarteiroes: {quantidade_de_quarteiroes:.0f}")
    print(f"Quantidade de cheeseburgers: {quantidade_de_cheeseburgers:.0f}")
    print(f"Libras de carne adicionais: {quantidade_de_libras_de_carne_adicionais:.0f}")
    print(f"Carne disponivel (inicial + comprada): {carne_disponivel:.2f} lb")
    print(f"Carne usada na producao: {carne_usada:.2f} lb")
    print(f"Excedente de carne (doacao): {excedente_carne:.2f} lb")
    print(f"Lucro: {lucro:.2f}")

    return ampl


ampl_mcburger = modelo_mcburger()

cbc 2.10.12:cbc 2.10.12: optimal solution; objective 209
0 simplex iterations
z = 209
q = 950
c = 0
x = 0

Quantidade de quarteiroes: 950
Quantidade de cheeseburgers: 0
Libras de carne adicionais: 0
Lucro: 209.00
